# Score-based search (XGES / hill climbing / Tabu search)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Abel-ai-lab/andrey/blob/main/examples/score_search.ipynb)

XGES, hill climbing, and Tabu search look for the graph with the best BIC score. XGES searches
over equivalence classes, as GES does, with deletions first. Hill climbing adds, removes, or
reverses one edge of a DAG at a time and stops when no move improves the score; Tabu search keeps
going past that point and returns the best DAG it visits. All three return a CPDAG and suit
linear-Gaussian data.

## Start

Choose a CPU runtime and run this cell first. In Colab it installs `andrey-core==0.6.0` from PyPI;
local kernels use the installed environment.

In [1]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    os.environ["ANDREY_DEVICE"] = "cpu"
    !pip install --quiet "andrey-core[numba,viz]==0.6.0" pandas

In [2]:
import numpy as np
import pandas as pd
from IPython.display import display

import andrey
import andrey.metrics as metrics
import andrey.viz
from andrey.data import benchmarks

## Data

`linear_gauss_er` draws an Erdos-Renyi graph with linear mechanisms and Gaussian noise: here 10
variables and 2,000 samples.

In [3]:
seed = 7
dataset = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=seed, scale="standardize")
X, truth = dataset.data, dataset.observed_graph
print(f"{X.shape[0]} observations, {X.shape[1]} variables, seed {seed}")
display(andrey.viz.draw(truth))

2000 observations, 10 variables, seed 7


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 464 330" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="464" height="330" fill="#fafbfc"/><line x1="144.33" y1="36.05" x2="410.18" y2="124.66" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.67,127.82 411.76,119.92 408.60,129.41" fill="#3a4658" stroke-linejoin="round"/><line x1="142.84" y1="39.11" x2="212.82" y2="85.38" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.16,90.89 215.57,81.21 210.06,89.55" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="44.62" x2="42.84" y2="90.89" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,39.11 110.06,40.45 115.57,48.79" fill="#3a4658" stroke-linejoin="round"/><line x1="212.53" y1="285.82" x2="143.00" y2="242.08" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.00,291.14 215.19,281.59 209.87,290.05" fill="#3a4658" stroke-linejoin="round"/><line x1="210.06" y1="291.16" x2="44.40" y2="239.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.60,294.16 211.56,286.39 208.56,295.93" fill="#3a4658" stroke-linejoin="round"/><line x1="144.93" y1="162.80" x2="409.13" y2="134.39" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.07,133.33 408.60,129.42 409.67,139.37" fill="#3a4658" stroke-linejoin="round"/><line x1="113.24" y1="177.50" x2="42.60" y2="227.64" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.40,171.72 110.35,173.43 116.14,181.58" fill="#3a4658" stroke-linejoin="round"/><line x1="112.82" y1="151.51" x2="42.84" y2="105.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.16,157.02 115.57,147.34 110.06,155.68" fill="#3a4658" stroke-linejoin="round"/><line x1="139.66" y1="224.66" x2="218.45" y2="116.65" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="224.34,108.57 214.41,113.70 222.49,119.59" fill="#3a4658" stroke-linejoin="round"/><line x1="109.00" y1="235.16" x2="45.00" y2="235.16" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,235.16 109.00,230.16 109.00,240.16" fill="#3a4658" stroke-linejoin="round"/><line x1="409.32" y1="128.10" x2="244.82" y2="100.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.18,129.76 410.16,123.17 408.49,133.02" fill="#3a4658" stroke-linejoin="round"/><line x1="410.11" y1="139.00" x2="344.37" y2="160.20" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.63,135.93 408.58,134.24 411.65,143.76" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="98.06" x2="45.00" y2="98.06" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,98.06 209.00,93.06 209.00,103.06" fill="#3a4658" stroke-linejoin="round"/><line x1="242.84" y1="105.24" x2="312.82" y2="151.51" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,157.02 315.57,147.34 310.06,155.68" fill="#3a4658" stroke-linejoin="round"/><line x1="242.84" y1="90.89" x2="312.82" y2="44.62" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.16,39.11 310.06,40.45 315.57,48.79" fill="#3a4658" stroke-linejoin="round"/><circle cx="132.00" cy="31.94" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="31.94" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="232.00" cy="298.06" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="298.06" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="132.00" cy="164.19" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="164.19" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominan

## Fit XGES, hill climbing, and Tabu search

`metadata["score"]` is the BIC score of the returned graph, which each search minimizes. A larger
`lambda_value` weighs the BIC penalty more and gives a sparser graph. With
`extended_search=False`, XGES skips its extended search for a faster fit, and `tabu_length` sets
how many recently visited DAGs Tabu search may not return to.

In [4]:
results = {
    "XGES": andrey.xges(X),
    "Hill climbing": andrey.hc(X),
    "Tabu search": andrey.tabu(X),
}
for name, result in results.items():
    print(f"{name}: {result.structure.kind}, score {result.metadata['score']:.1f}")
    display(andrey.viz.draw(result.structure))

XGES: cpdag, score -22406.2


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 364 340" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="364" height="340" fill="#fafbfc"/><line x1="209.00" y1="104.49" x2="45.00" y2="103.17" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,104.57 209.04,99.49 208.96,109.49" fill="#3a4658" stroke-linejoin="round"/><line x1="244.31" y1="108.85" x2="310.22" y2="131.17" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.69,134.38 311.82,126.43 308.61,135.91" fill="#3a4658" stroke-linejoin="round"/><line x1="217.58" y1="122.59" x2="140.15" y2="218.75" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="223.85,114.80 213.68,119.46 221.47,125.73" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="32.31" x2="109.00" y2="33.34" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.00,33.50 109.08,28.34 108.92,38.34" fill="#3a4658" stroke-linejoin="round"/><line x1="240.97" y1="43.12" x2="316.13" y2="121.91" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="323.03,129.14 319.74,118.45 312.51,125.36" fill="#3a4658" stroke-linejoin="round"/><line x1="209.00" y1="33.71" x2="145.00" y2="33.71" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,33.71 209.00,28.71 209.00,38.71" fill="#3a4658" stroke-linejoin="round"/><line x1="214.31" y1="48.41" x2="42.00" y2="191.53" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="222.00,42.02 211.11,44.56 217.50,52.25" fill="#3a4658" stroke-linejoin="round"/><line x1="40.09" y1="113.24" x2="117.69" y2="210.87" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="123.91,218.69 121.60,207.75 113.77,213.98" fill="#3a4658" stroke-linejoin="round"/><line x1="42.68" y1="95.66" x2="113.10" y2="46.82" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="121.32,41.12 110.25,42.71 115.95,50.93" fill="#3a4658" stroke-linejoin="round"/><line x1="311.04" y1="148.01" x2="143.85" y2="223.52" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="320.15,143.90 308.98,143.46 313.10,152.57" fill="#3a4658" stroke-linejoin="round"/><line x1="320.31" y1="158.35" x2="238.61" y2="296.71" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="325.39,149.74 316.00,155.81 324.61,160.90" fill="#3a4658" stroke-linejoin="round"/><line x1="109.91" y1="222.46" x2="44.48" y2="203.46" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="119.52,225.25 111.31,217.66 108.52,227.26" fill="#3a4658" stroke-linejoin="round"/><line x1="142.20" y1="236.93" x2="213.96" y2="293.64" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.80,299.84 217.06,289.72 210.85,297.56" fill="#3a4658" stroke-linejoin="round"/><line x1="144.96" y1="229.92" x2="209.07" y2="235.09" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.04,235.89 209.48,230.10 208.67,240.07" fill="#3a4658" stroke-linejoin="round"/><circle cx="232.00" cy="104.68" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="104.68" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="32.00" cy="32.10" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="32.10" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="232.00" cy="33.71" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="33.71" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="32.00" cy="103.06" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="103.06" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" d

Hill climbing: cpdag, score -22342.0


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 664 446" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="664" height="446" fill="#fafbfc"/><line x1="222.63" y1="326.99" x2="137.30" y2="135.68" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="226.70,336.13 227.20,324.96 218.06,329.03" fill="#3a4658" stroke-linejoin="round"/><line x1="242.84" y1="355.17" x2="321.16" y2="406.96" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="242.84" y1="340.83" x2="321.16" y2="289.04" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="209.04" y1="346.70" x2="44.98" y2="337.44" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.02,347.27 209.32,341.71 208.75,351.70" fill="#3a4658" stroke-linejoin="round"/><path d="M 244.64 344.96 Q 352.57 319.04 423.67 233.79" fill="none" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="41.57" y1="40.67" x2="115.07" y2="108.24" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="122.43,115.01 118.45,104.56 111.68,111.92" fill="#3a4658" stroke-linejoin="round"/><line x1="44.14" y1="36.53" x2="510.53" y2="215.56" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.86,219.15 512.32,210.90 508.74,220.23" fill="#3a4658" stroke-linejoin="round"/><line x1="139.37" y1="134.51" x2="318.95" y2="395.19" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="324.63,403.42 323.07,392.35 314.83,398.02" fill="#3a4658" stroke-linejoin="round"/><line x1="142.20" y1="131.87" x2="313.96" y2="267.61" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="321.80,273.81 317.06,263.69 310.85,271.53" fill="#3a4658" stroke-linejoin="round"/><line x1="122.22" y1="144.62" x2="37.53" y2="324.94" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="126.47,135.57 117.70,142.50 126.75,146.75" fill="#3a4658" stroke-linejoin="round"/><line x1="144.61" y1="126.96" x2="509.69" y2="218.23" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.39,220.65 510.90,213.38 508.47,223.08" fill="#3a4658" stroke-linejoin="round"/><line x1="143.92" y1="129.00" x2="610.91" y2="332.37" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="620.08,336.36 612.91,327.78 608.92,336.95" fill="#3a4658" stroke-linejoin="round"/><line x1="144.33" y1="127.92" x2="410.18" y2="216.53" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.67,219.70 411.76,211.79 408.60,221.28" fill="#3a4658" stroke-linejoin="round"/><line x1="309.73" y1="408.38" x2="44.59" y2="339.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.41,410.88 310.98,403.54 308.48,413.22" fill="#3a4658" stroke-linejoin="round"/><line x1="341.42" y1="405.17" x2="515.34" y2="239.66" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="522.58,232.77 511.89,236.04 518.79,243.28" fill="#3a4658" stroke-linejoin="round"/><line x1="344.64" y1="411.07" x2="609.64" y2="346.96" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.36,344.61 608.47,342.10 610.82,351.82" fill="#3a4658" stroke-linejoin="round"/><line x1="309.37" y1="286.01" x2="44.79" y2="334.37" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="319.21,284.21 308.48,281.09 310.27,290.93" fill="#3a4658" stroke-linejoin="round"/><line x1="343.24" y1="275.34" x2="420.76" y2="230.33" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><path d="M 44.96 335.69 Q 332.21 313.13 609.10 339.38" fill="none" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="619.06,340.32 609.57,334.40 608.63,344.36" fill="#3a4658" stroke-linejoin="round"/><line x1="44.51" y1="333.18" x2="409.86" y2="230.05" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.49,227.34 408.5

Tabu search: cpdag, score -22398.8


'<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 564 293" width="100%" font-family="Inter, system-ui, sans-serif"><rect width="564" height="293" fill="#fafbfc"/><line x1="42.84" y1="131.09" x2="121.16" y2="182.88" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="44.91" y1="122.42" x2="509.15" y2="68.51" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.09,67.35 508.58,63.54 509.73,73.47" fill="#3a4658" stroke-linejoin="round"/><line x1="45.00" y1="124.02" x2="219.00" y2="125.43" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="145.00" y1="261.02" x2="209.00" y2="261.02" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="219.00,261.02 209.00,256.02 209.00,266.02" fill="#3a4658" stroke-linejoin="round"/><line x1="439.96" y1="184.61" x2="517.91" y2="84.03" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="524.04,76.13 513.96,80.97 521.86,87.10" fill="#3a4658" stroke-linejoin="round"/><line x1="410.16" y1="202.11" x2="244.34" y2="256.94" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.66,198.97 408.59,197.36 411.73,206.85" fill="#3a4658" stroke-linejoin="round"/><line x1="409.00" y1="194.89" x2="345.00" y2="194.89" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="419.00,194.89 409.00,189.89 409.00,199.89" fill="#3a4658" stroke-linejoin="round"/><line x1="142.92" y1="183.00" x2="221.08" y2="132.58" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="142.60" y1="197.57" x2="213.24" y2="247.70" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="221.40,253.49 216.14,243.63 210.35,251.78" fill="#3a4658" stroke-linejoin="round"/><line x1="145.00" y1="190.36" x2="319.00" y2="194.57" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="509.44" y1="70.34" x2="244.75" y2="123.00" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.25,68.39 508.47,65.44 510.42,75.25" fill="#3a4658" stroke-linejoin="round"/><line x1="510.22" y1="58.48" x2="444.31" y2="36.15" stroke="#3a4658" stroke-width="2.0" stroke-linecap="round"/><polygon points="519.69,61.68 511.82,53.74 508.61,63.21" fill="#3a4658" stroke-linejoin="round"/><line x1="242.68" y1="132.94" x2="321.32" y2="187.48" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><path d="M 244.34 121.44 Q 337.51 90.53 420.95 38.83" fill="none" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><line x1="243.00" y1="118.61" x2="321.00" y2="69.55" stroke="#e0982b" stroke-width="2.0" stroke-linecap="round"/><circle cx="32.00" cy="123.92" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="32.00" y="123.92" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">0</text><circle cx="132.00" cy="261.02" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="261.02" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">1</text><circle cx="432.00" cy="194.89" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="432.00" y="194.89" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">2</text><circle cx="132.00" cy="190.05" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="132.00" y="190.05" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">3</text><circle cx="532.00" cy="65.85" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="532.00" y="65.85" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-baseline="central">4</text><circle cx="232.00" cy="125.53" r="13.00" fill="#ffffff" stroke="#182233" stroke-width="1.8"/><text x="232.00" y="125.53" text-anchor="middle" font-size="11" font-weight="600" fill="#182233" dominant-ba

## Compare with GES over ten datasets

Structural Hamming distance (SHD) counts errors against the truth's CPDAG; lower is better. The
empty graph's SHD is the number of true edges, so an estimate at or above it has not improved on
returning no edges. One dataset can favor any method, so the table covers ten seeds and their
median.

In [5]:
methods = {
    "GES": lambda X: andrey.ges(X),
    "XGES": lambda X: andrey.xges(X),
    "Hill climbing": lambda X: andrey.hc(X),
    "Tabu search": lambda X: andrey.tabu(X),
}
rows = {}
for s in range(10):
    draw = benchmarks.scm("linear_gauss_er", d=10).sample(n=2000, seed=s, scale="standardize")
    d = draw.observed_graph.n_nodes
    empty = andrey.GraphStructure.from_numpy(np.zeros((d, d), dtype=np.uint8), kind="cpdag")
    rows[s] = {"Empty graph": metrics.score(empty, draw.observed_graph)["shd"]}
    for name, fit in methods.items():
        rows[s][name] = metrics.score(fit(draw.data), draw.observed_graph)["shd"]
shd = pd.DataFrame(rows)
shd.columns.name = "seed"
shd["median"] = shd.median(axis=1)
shd

seed,0,1,2,3,4,5,6,7,8,9,median
Empty graph,15,15,15,15,15,15,15,15,15,15,15.0
GES,21,10,0,10,17,7,11,17,0,0,10.0
XGES,0,8,0,9,1,0,1,6,0,0,0.5
Hill climbing,22,22,0,19,1,13,22,22,19,4,19.0
Tabu search,21,0,0,9,1,0,16,11,22,4,6.5


Tabu search starts with hill climbing's moves, so its score is never worse than hill
climbing's on the same data. The [0.3.0 release card](https://andrey.abel.ai/docs/benchmarks/0.3.0.html)
compares XGES with the xges package and the other two with bnlearn, on larger graphs with ten rows
per variable.

## Use another dataset

Change `seed`, or the `d` and `n` passed to `benchmarks.scm` and `sample`, and rerun.